## Quick test of the new fcl before re-trying new eventViewer code
---

In the previous versions, `/exp/icarus/app/users/sdey2/NuGraphReco/cc1e0pi_v3/eventViewer`, an indexing error was spotted in that both truth and NuGraph2-predicted semantic labels looked off. Visually, it was clear that track-like and shower-like topologies were being mis-labelled. This was apparently a repurcussion of bypassing larsoft and trying to extract information directly from the stage1 root files where the event visualisation may work but the indexing of the labels do not necessarily directly coincide with the pfp or slc they appear to relate to. 

Using the following files as a reference: 
- Stage 1 root file (training sample itself: MPVMPR + BNB Overlays)
`/exp/icarus/app/users/sdey2/icaruscode-v10_06_00_04p04_production/nugraphProduction/localRunTest/MPVMPRTrainingDataset_MiniProductionTest/stage1-2dmodel-fixednorm-trainsample.root`
- FHiCL
`/exp/icarus/app/users/sdey2/icaruscode-v10_06_00_04p04_production/nugraphProduction/localRunTest/nugraph_analyser.fcl`
- `lar` command 
```
lar -c NuGraphAnalyser.fcl -s /exp/icarus/app/users/sdey2/icaruscode-v10_06_00_04p04_production/nugraphProduction/localRunTest/MPVMPRTrainingDataset_MiniProductionTest/stage1-2dmodel-fixednorm-trainsample.root -n 5
```
---

# NuGraph2 Predicted Event Viewer — MPVMPR (LArSoft dump)

Let's start here...

First let's read the output of `nugraph_analyser.fcl` (`ICARUSNuGraphAnalyzer`), which dumps the
2D NuGraph **predictions** per hit with art-native associations. Predictions only — no truth
in this file. Set `PRED_FILE` to your dump.

In [ ]:
%matplotlib inline
import uproot, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'font.family':'serif', 'font.serif':['Times New Roman','DejaVu Serif'],
    'mathtext.fontset':'cm', 'figure.dpi':130, 'figure.facecolor':'white',
    'axes.facecolor':'#f9f9f9', 'axes.grid':True, 'grid.color':'#e6e6e6',
    'axes.titlesize':11, 'axes.labelsize':11,
})

# -- input: the ICARUSNuGraphAnalyzer dump -------------------------------------
PRED_FILE = 'hist_nugraph_analyser_nue.root'   # <-- output of run_nugraph_analyzer.fcl 
CRYO      = 'CryoE'


# /exp/icarus/app/users/sdey2/icaruscode-v10_06_00_04p04_production/nugraphProduction/localRunTest/NuGraphAnalyzerChanges_Tests/hist_nugraph_analyser_trainingsample.root #--> training sample
# /exp/icarus/app/users/sdey2/icaruscode-v10_06_00_04p04_production/nugraphProduction/localRunTest/NuGraphAnalyzerChanges_Tests/hist_nugraph_analyser_bnboverlay.root #--> BNB sample
# /exp/icarus/app/users/sdey2/icaruscode-v10_06_00_04p04_production/nugraphProduction/localRunTest/NuGraphAnalyzerChanges_Tests/hist_nugraph_analyser_nue.root #--> nue only

SEM_LABELS  = ['MIP', 'HIP', 'shower', 'michel', 'diffuse']
SEM_COLORS  = ['#4575b4', '#d73027', '#1a9850', '#fdae61', '#984ea3']
SEM_CMAP    = dict(zip(SEM_LABELS, SEM_COLORS))
PLANE_NAMES = {0:'Induction-1', 1:'Induction-2', 2:'Collection'}

## Load

In [ ]:
def load_hits(path, cryo):
    df = uproot.open(path)[f'NuGraphAna{cryo}/NuGraphHitOutput'].arrays(library='pd')
    s = df[SEM_LABELS].values
    v = np.all(s >= 0, axis=1)
    df['sem_label'] = 'unknown'
    df.loc[v, 'sem_label'] = [SEM_LABELS[i] for i in np.argmax(s[v], axis=1)]
    df['sem_score'] = np.where(v, s.max(1), -1.0)
    df['pred']      = np.where(v, np.argmax(s, 1), -1)
    df['filt_pass'] = df['x_filter'] > 0.5          # NuGraph filter: signal vs noise
    return df

def event_list(df, max_events=None):
    ev = df[['run','subrun','event']].drop_duplicates().reset_index(drop=True)
    return ev if max_events is None else ev.head(max_events)

df = load_hits(PRED_FILE, CRYO)
print(f'{len(df)} hits | {CRYO} | events {sorted(df.event.unique())}')
for _, e in event_list(df).iterrows():
    g = df[df.event==e.event]
    ns = g[g.islc>=0].islc.nunique()
    print(f'  evt {int(e.event)}: {len(g):6d} hits, {ns} slices')

## Event overview & slice inventory

In [ ]:
def scan_overview(df, max_events=None):
    ev = event_list(df, max_events); n = len(ev)
    fig, axes = plt.subplots(n, 3, figsize=(13, 2.8*n), squeeze=False)
    for r in range(n):
        row = ev.iloc[r]
        v = df[(df.event==row.event) & (df.sem_score>=0)]
        for c in range(3):
            ax = axes[r][c]; d = v[v.plane==c]
            for lab in SEM_LABELS:
                l = d[d.sem_label==lab]
                if len(l): ax.scatter(l.wire,l.time,c=SEM_CMAP[lab],s=5,marker='s',lw=0)
            if r==0: ax.set_title(PLANE_NAMES[c])
            if c==0: ax.set_ylabel(f'evt {int(row.event)}\nTime',fontsize=8)
            if r==n-1: ax.set_xlabel('Wire')
    fig.legend(handles=[mpatches.Patch(color=SEM_CMAP[l],label=l) for l in SEM_LABELS],
               loc='upper center', ncol=5, bbox_to_anchor=(0.5,1.005))
    fig.suptitle('Predicted semantic scan -- '+CRYO, y=1.012)
    plt.tight_layout(); plt.show()

def slice_inventory(df, max_events=None):
    for _, e in event_list(df, max_events).iterrows():
        g = df[df.event==e.event]
        s = g[g.islc>=0].groupby('islc').size().sort_values(ascending=False)
        print(f' evt {int(e.event)}: {len(s)} slices | unsliced={int((g.islc<0).sum())} | '
              + ' '.join(f'[slc{int(i)}:{n}]' for i,n in s.items()))

def plot_slice(df, event, islc=None, planes=(0,1,2)):
    e = df[df.event==event]
    if islc is None:
        islc = int(e[e.islc>=0].groupby('islc').size().idxmax())
    dv = e[(e.islc==islc) & (e.sem_score>=0)]
    fig, axes = plt.subplots(1, len(planes), figsize=(4.4*len(planes),3.6), squeeze=False)
    for ax,c in zip(axes[0],planes):
        d = dv[dv.plane==c]
        for lab in SEM_LABELS:
            l = d[d.sem_label==lab]
            if len(l): ax.scatter(l.wire,l.time,c=SEM_CMAP[lab],s=8,marker='s',lw=0)
        ax.set_title(PLANE_NAMES[c]); ax.set_xlabel('Wire')
        if c==planes[0]: ax.set_ylabel('Time [tick]')
    fig.legend(handles=[mpatches.Patch(color=SEM_CMAP[l],label=l) for l in SEM_LABELS],
               loc='upper center', ncol=5, bbox_to_anchor=(0.5,1.06))
    fig.suptitle(f'{CRYO} | evt {event} | slice {islc} ({len(dv)} hits)', y=1.13)
    plt.tight_layout(); plt.show()

scan_overview(df)
print('=== slice inventory ===')
slice_inventory(df)
#plot_slice(df, 4)            # largest slice of event 4; set islc=<n> for any slice

In [ ]:
print(df.groupby('event').apply(lambda g: (g.islc>=0).sum()))
print('islc values:', sorted(df.islc.unique()))

## Predicted class composition

In [ ]:
def predicted_composition(df, per_event=False):
    d = df[df.sem_score>=0]
    fig, ax = plt.subplots(figsize=(7,4))
    if per_event:
        evs = sorted(d.event.unique()); x = np.arange(len(SEM_LABELS)); w=0.8/len(evs)
        for k,ev in enumerate(evs):
            f = d[d.event==ev]['sem_label'].value_counts().reindex(SEM_LABELS,fill_value=0).values.astype(float)
            f/=f.sum()
            ax.bar(x+(k-len(evs)/2)*w, f, w, label=f'evt {ev}')
        ax.set_xticks(x); ax.set_xticklabels(SEM_LABELS); ax.legend(fontsize=8)
    else:
        f = d['sem_label'].value_counts().reindex(SEM_LABELS,fill_value=0).values.astype(float); f/=f.sum()
        ax.bar(SEM_LABELS, f, color=[SEM_CMAP[l] for l in SEM_LABELS], edgecolor='k', lw=0.5)
        for i,v in enumerate(f): ax.text(i, v+0.005, f'{v*100:.0f}%', ha='center', fontsize=9)
    ax.set_ylabel('fraction of predicted hits')
    ax.set_title(f'Predicted class composition -- {CRYO} (all events)')
    plt.tight_layout(); plt.show()

predicted_composition(df)

## NuGraph filter score (signal vs noise)

In [ ]:
def filter_hist(df):
    x = df['x_filter'].values
    fig, ax = plt.subplots(figsize=(7,3.6))
    ax.hist(x, bins=50, color='#4575b4', edgecolor='white', lw=0.3)
    ax.axvline(0.5, color='#d73027', ls='--', lw=1.2, label='cut 0.5')
    ax.set_xlabel('NuGraph filter score  x_filter'); ax.set_ylabel('hits'); ax.legend()
    ax.set_title(f'Filter score -- {CRYO} | {100*np.mean(x>0.5):.0f}% pass (signal)')
    plt.tight_layout(); plt.show()

def filter_display(df, event, planes=(0,1,2)):
    e = df[df.event==event]
    fig, axes = plt.subplots(1, len(planes), figsize=(4.4*len(planes),3.6), squeeze=False)
    for ax,c in zip(axes[0],planes):
        d = e[e.plane==c]
        no = d[~d.filt_pass]; ye = d[d.filt_pass]
        if len(no): ax.scatter(no.wire,no.time,c='#cccccc',s=5,marker='s',lw=0,label='noise')
        if len(ye): ax.scatter(ye.wire,ye.time,c='#1a9850',s=5,marker='s',lw=0,label='signal')
        ax.set_title(PLANE_NAMES[c]); ax.set_xlabel('Wire')
        if c==planes[0]: ax.set_ylabel('Time [tick]')
    fig.legend(handles=[mpatches.Patch(color='#1a9850',label='signal (x_filter>0.5)'),
                        mpatches.Patch(color='#cccccc',label='noise')],
               loc='upper center', ncol=2, bbox_to_anchor=(0.5,1.08))
    fig.suptitle(f'NuGraph filter -- {CRYO} | evt {event}', y=1.14)
    plt.tight_layout(); plt.show()

filter_hist(df)
#filter_display(df, 4)